In [2]:
import pandas as pd
import numpy as np

class GranularWealthEngine:
    def __init__(self, inputs: dict, step_up_schedule: dict):
        """
        Initializes the model with custom inputs and an explicit manual step-up schedule.
        """
        self.client_name = inputs.get("client_name", "Valued Client")
        self.current_age = inputs.get("current_age", 35)
        self.annual_premium = inputs.get("annual_premium", 150000)
        self.payout_pct = inputs.get("payout_pct", 0.40)
        self.life_cover_multiple = inputs.get("life_cover_multiple", 7)
        self.ppt = inputs.get("ppt", 12)
        self.policy_term = inputs.get("policy_term", 40)
        self.expected_return = inputs.get("expected_return", 0.18)
        self.monthly_swp_target = inputs.get("monthly_swp", 100000)
        self.swp_start_age = inputs.get("swp_start_age", 60)
        
        # User defined structural schedule: { Policy_Year: Absolute_Monthly_Increase_Amount }
        self.step_up_schedule = step_up_schedule
        
        # Derived Base Variables
        self.life_cover = self.annual_premium * self.life_cover_multiple
        self.annual_payout = self.annual_premium * self.payout_pct
        self.base_monthly_sip = round(self.annual_payout / 12, 2)
        self.monthly_rate = self.expected_return / 12
        
    def run_projection(self) -> pd.DataFrame:

        records = []
        current_corpus = 0.0
        
        # We track running cumulative step-up additions across time
        cumulative_step_up = 0.0
        
        for year in range(1, self.policy_term + 1):
            age = self.current_age + (year - 1)
            
            # 1. Base Premium and Payout rules
            premium_paid = self.annual_premium if year <= self.ppt else 0.0
            insurance_payout = self.annual_payout
            
            # 2. GRANULAR LOGIC: Check if this specific year triggers a fresh manual increase
            # If Year 3 has a 100 step-up, it permanently lifts the floor by 100 for Year 3 and beyond.
            year_specific_increment = self.step_up_schedule.get(year, 0.0)
            cumulative_step_up += year_specific_increment
            
            total_monthly_sip = self.base_monthly_sip + cumulative_step_up
            annual_sip_contribution = total_monthly_sip * 12
            
            # 3. Monthly Financial Compounding Loop
            monthly_sip = total_monthly_sip
            monthly_swp = self.monthly_swp_target if age >= self.swp_start_age else 0.0
            
            for month in range(1, 13):
                if current_corpus <= 0:
                    current_corpus = 0.0
                
                # Add monthly investment & compound
                current_corpus += monthly_sip
                current_corpus *= (1 + self.monthly_rate)
                
                # Deduct monthly withdrawal if active
                if current_corpus >= monthly_swp:
                    current_corpus -= monthly_swp
                else:
                    current_corpus = 0.0
            
            # End of Year Corpus Value
            net_corpus = round(current_corpus, 2)
            annual_swp_withdrawn = (self.monthly_swp_target * 12) if age >= self.swp_start_age else 0.0
            
            # 4. Sustainability check
            status = "Corpus Exhausted" if net_corpus <= 0 else "Sustainable"
                
            records.append({
                "Policy Year": year,
                "Age": age,
                "Premium Paid (₹)": premium_paid,
                "Insurance Payout (₹)": insurance_payout,
                "Base Monthly SIP (₹)": self.base_monthly_sip,
                "New Step-Up Added (₹)": year_specific_increment,
                "Total Monthly SIP (₹)": total_monthly_sip,
                "Annual SIP Contribution (₹)": annual_sip_contribution,
                "Net End-of-Year Corpus (₹)": net_corpus,
                "Annual SWP Withdrawal (₹)": annual_swp_withdrawn,
                "Sustainability Flag": status
            })
            
        return pd.DataFrame(records)

    def generate_executive_summary(self, df: pd.DataFrame) -> dict:
        """
        Parses the matrix to generate advisor KPI metrics.
        """
        retirement_row = df[df["Age"] == self.swp_start_age]
        corpus_at_retirement = retirement_row["Net End-of-Year Corpus (₹)"].values[0] if not retirement_row.empty else 0.0
        
        final_corpus = df["Net End-of-Year Corpus (₹)"].iloc[-1]
        
        r_monthly = self.monthly_rate
        n_months = (self.policy_term - (self.swp_start_age - self.current_age)) * 12
        required_corpus = self.monthly_swp_target * ((1 - (1 + r_monthly)**-n_months) / r_monthly) if n_months > 0 else 0.0
            
        exhausted_rows = df[df["Sustainability Flag"] == "Corpus Exhausted"]
        if not exhausted_rows.empty:
            survival_status = "Corpus Exhausted"
            survives_until_year = exhausted_rows["Policy Year"].values[0]
            survives_until_age = exhausted_rows["Age"].values[0]
        else:
            survival_status = "Sustainable"
            survives_until_year = self.policy_term
            survives_until_age = self.current_age + self.policy_term
            
        return {
            "Life Cover Amount": self.life_cover,
            "Monthly Insurance Payout": self.base_monthly_sip,
            "Corpus at SWP Start": corpus_at_retirement,
            "Target Required Corpus": round(required_corpus, 2),
            "Surplus / Shortfall": round(corpus_at_retirement - required_corpus, 2),
            "Final Year 40 Corpus": final_corpus,
            "Sustainability Status": survival_status,
            "Years Corpus Survives": survives_until_year,
            "Age Corpus Exhausted": survives_until_age
        }


if __name__ == "__main__":
    
    # Please make all the changes in this section according to the clients
    advisor_inputs = {
        "client_name": "Aditya Sharma",
        "current_age": 40,
        "annual_premium": 150000,
        "payout_pct": 0.40,
        "life_cover_multiple": 7,
        "ppt": 12,
        "policy_term": 40,
        "expected_return": 0.18, 
        "monthly_swp": 125000,   
        "swp_start_age": 60      
    }

    # This is for the set-up SIP
    
    custom_schedule = {       #Year: Absolute Monthly Increase in SIP

    }

    # Fire the calculation architecture
    engine = GranularWealthEngine(advisor_inputs, custom_schedule)
    projection_matrix = engine.run_projection()
    summary = engine.generate_executive_summary(projection_matrix)

    # 1. Print Executive Summary Panel
    print(f"\n" + "═"*75)
    print(f" WEALTH PLANNER EXECUTIVE DASHBOARD: {engine.client_name.upper()}")
    print("═"*75)
    print(f" Initial Monthly Base Payout     : ₹{summary['Monthly Insurance Payout']:,.2f}")
    print(f" Client Custom Schedule Rules     : {custom_schedule}")
    print(f" Total Life Cover Provided       : ₹{summary['Life Cover Amount']:,.2f}")
    print(f" Capital Balance at Age 60        : ₹{summary['Corpus at SWP Start']:,.2f}")
    print(f" Required Balance Target for SWP  : ₹{summary['Target Required Corpus']:,.2f}")
    print(f" Calculated Surplus / Shortfall  : ₹{summary['Surplus / Shortfall']:,.2f}")
    print(f" Terminal Portfolio Value (Yr 40): ₹{summary['Final Year 40 Corpus']:,.2f}")
    print(f" Strategy Sustainability Status  : {summary['Sustainability Status'].upper()}")
    if summary['Sustainability Status'] == "Corpus Exhausted":
        print(f" 🚨 ALERT: Running out of cash in Policy Year {summary['Years Corpus Survives']} (Age {summary['Age Corpus Exhausted']})")
    print("═"*75 + "\n")

    # 2. Print Every Column Across the Entire 40-Year Horizon
    print(">>> COMPLETE 40-YEAR AUDIT LEDGER MATRIX:")
    
    # Selecting all columns generated in the simulation frame
    all_cols = [
        "Policy Year", 
        "Age", 
        "Premium Paid (₹)", 
        "Insurance Payout (₹)", 
        "Base Monthly SIP (₹)", 
        "New Step-Up Added (₹)", 
        "Total Monthly SIP (₹)", 
        "Annual SIP Contribution (₹)", 
        "Annual SWP Withdrawal (₹)",
        "Net End-of-Year Corpus (₹)", 
        "Sustainability Flag"
    ]
    
    # Maximize layout context width to prevent string column wrapping in terminal windows
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    pd.set_option('display.max_rows', 50)
    
    # Print the entire structure seamlessly without indexing breaks
    print(projection_matrix[all_cols].to_string(index=False))


═══════════════════════════════════════════════════════════════════════════
 WEALTH PLANNER EXECUTIVE DASHBOARD: ADITYA SHARMA
═══════════════════════════════════════════════════════════════════════════
 Initial Monthly Base Payout     : ₹5,000.00
 Client Custom Schedule Rules     : {}
 Total Life Cover Provided       : ₹1,050,000.00
 Capital Balance at Age 60        : ₹12,445,612.04
 Required Balance Target for SWP  : ₹8,099,466.51
 Calculated Surplus / Shortfall  : ₹4,346,145.53
 Terminal Portfolio Value (Yr 40): ₹140,635,872.87
 Strategy Sustainability Status  : SUSTAINABLE
═══════════════════════════════════════════════════════════════════════════

>>> COMPLETE 40-YEAR AUDIT LEDGER MATRIX:
 Policy Year  Age  Premium Paid (₹)  Insurance Payout (₹)  Base Monthly SIP (₹)  New Step-Up Added (₹)  Total Monthly SIP (₹)  Annual SIP Contribution (₹)  Annual SWP Withdrawal (₹)  Net End-of-Year Corpus (₹) Sustainability Flag
           1   40          150000.0               60000.0         

In [18]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

def export_to_styled_excel(df: pd.DataFrame, summary_data: dict, filename="Retirement_Strategy_Dashboard.xlsx"):
    """
    Ingests the projection dataframe and writes a highly styled, 
    wealth-planner-quality Excel spreadsheet.
    """
    # Create empty workbook and setup active sheet
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Retirement Strategy"
    
    # Grid lines must be visible for clean presentation
    ws.views.sheetView[0].showGridLines = True
    
    # ════════════════════════════════════
    # PALETTE & TYPOGRAPHY SYSTEM
    # ════════════════════════════════════
    FONT_FAMILY = "Segoe UI"
    
    # Colors
    NAVY_DARK = "1B365D"    # Primary headers / Title Block
    NAVY_LIGHT = "F0F4F8"   # Zebra striping
    ACCENT_BLUE = "4A90E2"  # Highlight elements
    KPI_BG = "FAFAFA"       # Summary card fill
    WHITE = "FFFFFF"
    
    # Status Alert Colors
    GREEN_FILL = "D4EDDA"
    GREEN_TEXT = "155724"
    RED_FILL = "F8D7DA"
    RED_TEXT = "721C24"

    # Styles
    title_font = Font(name=FONT_FAMILY, size=18, bold=True, color=WHITE)
    section_font = Font(name=FONT_FAMILY, size=12, bold=True, color=NAVY_DARK)
    header_font = Font(name=FONT_FAMILY, size=10, bold=True, color=WHITE)
    bold_font = Font(name=FONT_FAMILY, size=10, bold=True)
    regular_font = Font(name=FONT_FAMILY, size=10)
    
    # Fills
    title_fill = PatternFill(start_color=NAVY_DARK, end_color=NAVY_DARK, fill_type="solid")
    header_fill = PatternFill(start_color=NAVY_DARK, end_color=NAVY_DARK, fill_type="solid")
    zebra_fill = PatternFill(start_color=NAVY_LIGHT, end_color=NAVY_LIGHT, fill_type="solid")
    kpi_fill = PatternFill(start_color=KPI_BG, end_color=KPI_BG, fill_type="solid")
    
    # Borders
    thin_border_side = Side(border_style="thin", color="D3D3D3")
    thin_border = Border(left=thin_border_side, right=thin_border_side, top=thin_border_side, bottom=thin_border_side)
    thick_bottom = Border(bottom=Side(border_style="medium", color=NAVY_DARK))
    double_bottom = Border(top=thin_border_side, bottom=Side(border_style="double", color=NAVY_DARK))

    # Alignments
    align_center = Alignment(horizontal="center", vertical="center", wrap_text=True)
    align_left = Alignment(horizontal="left", vertical="center")
    align_right = Alignment(horizontal="right", vertical="center")

    # ════════════════════════════════════
    # 1. HEADER BANNER BLOCK
    # ════════════════════════════════════
    ws.merge_cells("A1:K2")
    title_cell = ws["A1"]
    title_cell.value = "  STRATEGIC RETIREMENT & REINVESTMENT ARCHITECTURE"
    title_cell.font = title_font
    title_cell.fill = title_fill
    title_cell.alignment = Alignment(horizontal="left", vertical="center")
    
    # Fix merged backgrounds
    for row in ws["A1:K2"]:
        for cell in row:
            cell.fill = title_fill

    # ════════════════════════════════════
    # 2. KPI / SUMMARY OVERVIEW CARDS
    # ════════════════════════════════════
    ws["A4"] = "PLAN METRICS & PERFORMANCE TARGETS"
    ws["A4"].font = section_font
    
    kpis = [
        ("Life Cover Amount", summary_data["Life Cover Amount"], '"₹"#,##,##0.00', "B", "C"),
        ("Monthly Base Payout", summary_data["Monthly Insurance Payout"], '"₹"#,##,##0.00', "B", "C"),
        ("Corpus at Age 60", summary_data["Corpus at SWP Start"], '"₹"#,##,##0.00', "E", "F"),
        ("Target Needed for SWP", summary_data["Target Required Corpus"], '"₹"#,##,##0.00', "E", "F"),
        ("Surplus / Shortfall", summary_data["Surplus / Shortfall"], '"₹"#,##,##0.00', "H", "I"),
        ("Terminal Year 40 Corpus", summary_data["Final Year 40 Corpus"], '"₹"#,##,##0.00', "H", "I"),
    ]
    
    # Draw KPI Layout blocks
    row_cursor = 5
    for label, val, num_fmt, start_col, end_col in kpis:
        # We alternate layout lines to form clean cards side by side
        c1 = ws[f"{start_col}{row_cursor}"]
        c2 = ws[f"{end_col}{row_cursor}"]
        
        c1.value = label
        c1.font = regular_font
        c1.fill = kpi_fill
        c1.border = thin_border
        
        c2.value = val
        c2.font = bold_font
        c2.fill = kpi_fill
        c2.border = thin_border
        c2.number_format = num_fmt
        c2.alignment = align_right
        
        row_cursor = row_cursor + 1 if start_col == "H" else row_cursor
    
    # Clean step adjustment for row index tracking
    row_cursor = 8
    
    # Sustainability Status Badge
    ws["B8"] = "Strategy Status"
    ws["B8"].font = regular_font
    ws["C8"].value = summary_data["Sustainability Status"].upper()
    ws["C8"].alignment = align_center
    ws["B8"].border = thin_border
    ws["C8"].border = thin_border
    
    if "SUSTAINABLE" in summary_data["Sustainability Status"].upper():
        ws["C8"].fill = PatternFill(start_color=GREEN_FILL, end_color=GREEN_FILL, fill_type="solid")
        ws["C8"].font = Font(name=FONT_FAMILY, size=10, bold=True, color=GREEN_TEXT)
    else:
        ws["C8"].fill = PatternFill(start_color=RED_FILL, end_color=RED_FILL, fill_type="solid")
        ws["C8"].font = Font(name=FONT_FAMILY, size=10, bold=True, color=RED_TEXT)

    # ════════════════════════════════════
    # 3. 40-YEAR DATA LEDGER TABLE
    # ════════════════════════════════════
    data_start_row = 11
    ws.cell(row=data_start_row-1, column=1, value="CHRONOLOGICAL AUDIT LEDGER").font = section_font
    
    headers = list(df.columns)
    
    # Write Table Headers
    for col_idx, header in enumerate(headers, start=1):
        cell = ws.cell(row=data_start_row, column=col_idx, value=header)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = align_center
        cell.border = thin_border
    
    ws.row_dimensions[data_start_row].height = 28
    
    # Write Rows Data
    current_row = data_start_row + 1
    for r_idx, row_data in df.iterrows():
        ws.row_dimensions[current_row].height = 20
        is_zebra = (r_idx % 2 != 0)
        
        for col_idx, value in enumerate(row_data, start=1):
            cell = ws.cell(row=current_row, column=col_idx, value=value)
            cell.font = regular_font
            cell.border = thin_border
            
            if is_zebra:
                cell.fill = zebra_fill
                
            # Alignment & Contextual Formatting rules based on type
            if isinstance(value, (int, float, np.number)):
                cell.alignment = align_right
                if col_idx in [1, 2]: # Policy Year and Age
                    cell.alignment = align_center
                    cell.number_format = '0'
                else:
                    cell.number_format = '"₹"#,##,##0.00'
            else:
                cell.alignment = align_center
                # Stylize row explicit status flag values dynamically
                if str(value) == "Sustainable":
                    cell.fill = PatternFill(start_color=GREEN_FILL, end_color=GREEN_FILL, fill_type="solid")
                    cell.font = Font(name=FONT_FAMILY, size=10, color=GREEN_TEXT, bold=True)
                elif str(value) == "Corpus Exhausted":
                    cell.fill = PatternFill(start_color=RED_FILL, end_color=RED_FILL, fill_type="solid")
                    cell.font = Font(name=FONT_FAMILY, size=10, color=RED_TEXT, bold=True)
                    
        current_row += 1

    # ════════════════════════════════════
    # 4. AUTO-FIT COLUMN COLS WIDTHS
    # ════════════════════════════════════
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        
        # Skip evaluating merged banner string length to safeguard baseline grid scales
        for cell in col:
            if cell.row > 2 and cell.value:
                max_len = max(max_len, len(str(cell.value)))
                
        ws.column_dimensions[col_letter].width = max(max_len + 4, 13)

    # Save output to spreadsheet asset
    wb.save(filename)
    print(f"✔️ Workbook compiled successfully as: '{filename}'")

# ══════════════════════════════════════════════════════════════════════════════
# COMPILER RUNTIME ENTRYPOINT
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # Ingest baseline simulation structures 
    engine = GranularWealthEngine(advisor_inputs, custom_schedule)
    projection_matrix = engine.run_projection()
    summary = engine.generate_executive_summary(projection_matrix)
    
    # Export fully styled spreadsheet representation to file disk
    export_to_styled_excel(projection_matrix, summary, "Retirement_Strategy_Dashboard.xlsx")

✔️ Workbook compiled successfully as: 'Retirement_Strategy_Dashboard.xlsx'
